In [ ]:
%pip install pandas

# Task 1

In [1]:
import pandas as pd
import os
directory = os.getcwd()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

In [2]:
data_directory = os.path.join(directory, 'data')
forecasts_export = pd.read_csv(os.path.join(data_directory, 'forecasts_export.csv'))
forecaster_demographics = pd.read_csv(os.path.join(data_directory, 'forecaster_demographics.csv'))
question_tags = pd.read_csv(os.path.join(data_directory, 'question_tags.csv'))

In [3]:
print("Question Tags Shape:", question_tags.shape)
print("Forecaster Demographics Shape:", forecaster_demographics.shape)
print("Forecasts Export Shape:", forecasts_export.shape)

Question Tags Shape: (20, 2)
Forecaster Demographics Shape: (11, 6)
Forecasts Export Shape: (90, 8)


In [4]:
question_tags.groupby('question_id')['tag'].apply(list).reset_index()

,question_id,tag
0,Q001,"[artificial-intelligence, benchmarks, capabilities, artificial-intelligence]"
1,Q002,"[artificial-intelligence, compute]"
2,Q003,"[energy, Artificial-Intelligence, infrastructure]"
3,Q004,"[scientific-discovery, artificial-intelligence]"
4,Q005,"[safety, artificial-intelligence, governance]"
5,Q006,"[artificial-intelligence, mathematics]"
6,Q007,"[labor-market, governance]"
7,Q008,"[regulation, governance]"


We can see that Q001 has two artificial-intelligence tags and there are some Artificial-Intelligence tags which needs to be lowercase to keep data consistant

In [5]:
forecaster_demographics.head(2)

,forecaster_id,years_forecasting_experience,education_level,country,affiliation,joined_date
0,F001,8,PhD,United States,University of Atlantis,2022-06-15
1,F002,5,Masters,United Kingdom,Bigfoot Institute of Technology,2023-01-20


In [6]:
forecaster_demographics.info()

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   forecaster_id                 11 non-null     str  
 1   years_forecasting_experience  11 non-null     int64
 2   education_level               10 non-null     str  
 3   country                       11 non-null     str  
 4   affiliation                   10 non-null     str  
 5   joined_date                   11 non-null     str  
dtypes: int64(1), str(5)
memory usage: 660.0 bytes


In [7]:
forecaster_demographics[forecaster_demographics['education_level'].isnull()]

,forecaster_id,years_forecasting_experience,education_level,country,affiliation,joined_date
6,F007,2,NaN,Japan,NaN,2024-01-10


In [8]:
forecaster_demographics['joined_date']

0         2022-06-15
1         2023-01-20
2         2021-09-01
3         2023-08-10
4         2023-03-15
5     March 15, 2023
6         2024-01-10
7         2022-03-01
8         2022-11-15
9         2021-05-20
10        2024-02-28
Name: joined_date, dtype: str

We can see F007 has NaN values so we need to keep education_level and affiliation optional (we can also flag it for further decision) and  March 15, 2023 needs to be properly formatted in date

In [9]:
forecasts_export.head(2)

,forecaster_id,forecaster_name,forecaster_type,question_id,question_text,forecast_timestamp,prediction,rationale
0,F001,Fish McWhaley,superforecaster,Q001,Will AI systems achieve >90% on Humanity's Last Exam by 2030?,2025-01-15 09:30:00,25,AI progress is impressive but Humanity's Last Exam is specifically designed to be extremely difficult. Incremental gains on benchmarks don't easily extrapolate to near-perfect performance.
1,F001,Fish McWhaley,superforecaster,Q001,Will AI systems achieve >90% on Humanity's Last Exam by 2030?,2025-03-20 14:15:00,30,Updating upward after seeing recent benchmark results. Still skeptical but progress is faster than I expected.


In [10]:
forecasts_export.info()

<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   forecaster_id       90 non-null     str  
 1   forecaster_name     90 non-null     str  
 2   forecaster_type     90 non-null     str  
 3   question_id         90 non-null     str  
 4   question_text       90 non-null     str  
 5   forecast_timestamp  90 non-null     str  
 6   prediction          90 non-null     int64
 7   rationale           87 non-null     str  
dtypes: int64(1), str(7)
memory usage: 5.8 KB


In [11]:
forecasts_export.isnull().sum()

forecaster_id         0
forecaster_name       0
forecaster_type       0
question_id           0
question_text         0
forecast_timestamp    0
prediction            0
rationale             3
dtype: int64

I beleive rationale is very important so lets flag these null rows also. I am not removing any row since prediction is very much required.

In [12]:
forecasts_export.groupby('forecaster_id')['forecaster_name'].nunique().reset_index()

,forecaster_id,forecaster_name
0,F001,1
1,F002,1
2,F003,1
3,F004,1
4,F005,1
5,F006,1
6,F007,1
7,F008,1
8,F009,1
9,F010,1


In [13]:
forecasts_export.groupby('forecaster_name')['forecaster_type'].nunique().reset_index()

,forecaster_name,forecaster_type
0,Blaze Thunderton,1
1,Coral Reefington,1
2,Fish McWhaley,1
3,Maple Syrupsworth,1
4,Noodle VonCrunch,1
5,Pepper Mintstick,1
6,Pickle Moonbeam,1
7,Rocket Soupkins,1
8,Toast Butterworth,1
9,Waffle Ironside,1


In [14]:
forecasts_export.groupby('question_id')['question_text'].nunique().reset_index()

,question_id,question_text
0,Q001,1
1,Q002,1
2,Q003,1
3,Q004,1
4,Q005,1
5,Q006,1
6,Q007,1
7,Q008,1


Everything else seems correct

# Task 2

In [15]:
import duckdb
import pandas as pd
con = duckdb.connect("fri_worktest.duckdb")

In [16]:
# See tables in the database
con.execute("SHOW TABLES").fetchdf()

,name
0,raw_forecaster_demographics
1,raw_forecasts_export
2,raw_question_tags


In [17]:
# reading tables from the database
question_tags_table = con.execute("SELECT * FROM question_tags").fetchdf()
forecaster_demographics_table = con.execute("SELECT * FROM forecaster_demographics").fetchdf()
forecasts_export_table = con.execute("SELECT * FROM forecasts_export").fetchdf()

In [18]:
question_tags_table.shape

(20, 2)

### question_tags_table

In [19]:
## Create a dimension table for questions with their associated tags
dim_questions_df = question_tags_table.groupby('question_id')['tag'].apply(list).reset_index()
print("Shape of dimension table for questions after grouping:", dim_questions_df.shape)
## Convert tags to lowercase and remove duplicates
dim_questions_df['tags'] = dim_questions_df['tag'].apply(lambda tags: list(set([tag.lower() for tag in tags])))
dim_questions_df.drop('tag', axis=1, inplace=True)
print("Shape of dimension table for questions after cleaning:", dim_questions_df.shape)
## Merge with forecasts_export to get question_text (because question_id is the key in both tables)
dim_questions_df = forecasts_export_table[['question_id', 'question_text']].drop_duplicates().merge(dim_questions_df, on='question_id', how='left')
print("Shape of dimension table for questions after merging with forecasts:", dim_questions_df.shape)
print("Any null values in tags column?", dim_questions_df.isnull().sum())
dim_questions_df.head(2)

Shape of dimension table for questions after grouping: (8, 2)
Shape of dimension table for questions after cleaning: (8, 2)
Shape of dimension table for questions after merging with forecasts: (8, 3)
Any null values in tags column? question_id      0
question_text    0
tags             0
dtype: int64


,question_id,question_text,tags
0,Q001,Will AI systems achieve >90% on Humanity's Last Exam by 2030?,"[benchmarks, artificial-intelligence, capabilities]"
1,Q002,Will a single AI training run exceed 1e28 FLOP by 2027?,"[artificial-intelligence, compute]"


### forecaster_demographics_table

In [20]:
## Change joined_date to datetime format
forecaster_demographics_df = forecaster_demographics_table.copy(deep=True)
forecaster_demographics_df['joined_date'] = pd.to_datetime(forecaster_demographics_df['joined_date'], format='mixed')
## merge with forecasts_export to get forecaster_name and forecaster_type (because forecaster_id is the key in both tables)
dim_forecasters_df = forecasts_export_table[['forecaster_id', 'forecaster_name', 'forecaster_type']].drop_duplicates().merge(forecaster_demographics_df, on='forecaster_id', how='left')
print("Shape of dimension table for forecasters after merging with forecasts:", dim_forecasters_df.shape)
print("Any null values in the dimension table for forecasters?\n", dim_forecasters_df.isnull().sum())
print("Data types in the dimension table for forecasters:\n", dim_forecasters_df.dtypes)
dim_forecasters_df.head(2)

Shape of dimension table for forecasters after merging with forecasts: (10, 8)
Any null values in the dimension table for forecasters?
 forecaster_id                   0
forecaster_name                 0
forecaster_type                 0
years_forecasting_experience    0
education_level                 1
country                         0
affiliation                     1
joined_date                     0
dtype: int64
Data types in the dimension table for forecasters:
 forecaster_id                              str
forecaster_name                            str
forecaster_type                            str
years_forecasting_experience             int64
education_level                            str
country                                    str
affiliation                                str
joined_date                     datetime64[us]
dtype: object


,forecaster_id,forecaster_name,forecaster_type,years_forecasting_experience,education_level,country,affiliation,joined_date
0,F001,Fish McWhaley,superforecaster,8,PhD,United States,University of Atlantis,2022-06-15
1,F002,Coral Reefington,superforecaster,5,Masters,United Kingdom,Bigfoot Institute of Technology,2023-01-20


### forecasts_export_table

In [21]:
forecasts_export_table.head(2)
forecasts_export_df = forecasts_export_table.copy(deep=True)
print("Shape of fact table for forecasts before dropping redundant columns:", forecasts_export_df.shape)
## remove question_text, forecaster_type and forecaster_name columns to avoid redundancy (because we have dimension tables for questions and forecasters now)
forecasts_export_df.drop(['question_text', 'forecaster_name', 'forecaster_type'], axis=1, inplace=True)
print("Shape of fact table for forecasts after dropping redundant columns:", forecasts_export_df.shape)
## change forecast_timestamp to datetime format
forecasts_export_df['forecast_timestamp'] = pd.to_datetime(forecasts_export_df['forecast_timestamp'], format='mixed')
print("Any null values in the fact table for forecasts?\n", forecasts_export_df.isnull().sum())
print("Data types in the fact table for forecasts:\n", forecasts_export_df.dtypes)
forecasts_export_df.head(2)

Shape of fact table for forecasts before dropping redundant columns: (90, 8)
Shape of fact table for forecasts after dropping redundant columns: (90, 5)
Any null values in the fact table for forecasts?
 forecaster_id         0
question_id           0
forecast_timestamp    0
prediction            0
rationale             3
dtype: int64
Data types in the fact table for forecasts:
 forecaster_id                    str
question_id                      str
forecast_timestamp    datetime64[us]
prediction                     int64
rationale                        str
dtype: object


,forecaster_id,question_id,forecast_timestamp,prediction,rationale
0,F001,Q001,2025-01-15 09:30:00,25,AI progress is impressive but Humanity's Last Exam is specifically designed to be extremely difficult. Incremental gains on benchmarks don't easily extrapolate to near-perfect performance.
1,F001,Q001,2025-03-20 14:15:00,30,Updating upward after seeing recent benchmark results. Still skeptical but progress is faster than I expected.


## Now lets create dim_forecasters, dim_questions and fct_forecasts with forign keys

In [22]:
print("dtypes in dimension table for questions:\n", dim_questions_df.dtypes)
print("dtypes in dimension table for forecasters:\n", dim_forecasters_df.dtypes)
print("dtypes in fact table for forecasts:\n", forecasts_export_df.dtypes)

dtypes in dimension table for questions:
 question_id         str
question_text       str
tags             object
dtype: object
dtypes in dimension table for forecasters:
 forecaster_id                              str
forecaster_name                            str
forecaster_type                            str
years_forecasting_experience             int64
education_level                            str
country                                    str
affiliation                                str
joined_date                     datetime64[us]
dtype: object
dtypes in fact table for forecasts:
 forecaster_id                    str
question_id                      str
forecast_timestamp    datetime64[us]
prediction                     int64
rationale                        str
dtype: object


In [23]:
forecasts_export_df['forecast_id'] = ["p" + str(i).zfill(3) for i in range(forecasts_export_df.shape[0])]
forecasts_export_df.head(2)

,forecaster_id,question_id,forecast_timestamp,prediction,rationale,forecast_id
0,F001,Q001,2025-01-15 09:30:00,25,AI progress is impressive but Humanity's Last Exam is specifically designed to be extremely difficult. Incremental gains on benchmarks don't easily extrapolate to near-perfect performance.,p000
1,F001,Q001,2025-03-20 14:15:00,30,Updating upward after seeing recent benchmark results. Still skeptical but progress is faster than I expected.,p001


In [24]:
con.execute(f"DROP TABLE IF EXISTS fct_forecasts")
con.execute(f"DROP TABLE IF EXISTS dim_questions")
con.execute(f"DROP TABLE IF EXISTS dim_forecasters")

con.execute("""CREATE TABLE dim_questions (
    question_id VARCHAR(4) PRIMARY KEY,
    question_text TEXT,
    tags VARCHAR(255)[]
)""")
con.execute("""CREATE TABLE dim_forecasters (
    forecaster_id VARCHAR(4) PRIMARY KEY,
    forecaster_name VARCHAR(255),
    forecaster_type VARCHAR(255),
    years_forecasting_experience INT,
    education_level VARCHAR(255),
    country VARCHAR(255),
    affiliation VARCHAR(255),
    joined_date DATE
)""")
con.execute("""CREATE TABLE fct_forecasts (
    forecast_id VARCHAR(4) PRIMARY KEY,
    question_id VARCHAR(4),
    forecaster_id VARCHAR(4),
    prediction INT,
    forecast_timestamp TIMESTAMP,
    rationale TEXT,
    FOREIGN KEY (question_id) REFERENCES dim_questions(question_id),
    FOREIGN KEY (forecaster_id) REFERENCES dim_forecasters(forecaster_id)
)""")

con.execute("INSERT INTO dim_questions SELECT * FROM dim_questions_df")
con.execute("INSERT INTO dim_forecasters SELECT * FROM dim_forecasters_df")
con.execute("""
    INSERT INTO fct_forecasts
    SELECT
        forecast_id        AS forecast_id,
        question_id        AS question_id,
        forecaster_id      AS forecaster_id,
        prediction         AS prediction,
        forecast_timestamp AS forecast_timestamp,
        rationale          AS rationale
    FROM forecasts_export_df
""")

In [25]:
con.close()

# Task 3

In [26]:
import duckdb
import pandas as pd
con = duckdb.connect("fri_worktest.duckdb")
con.execute("SHOW TABLES").fetchdf()

,name
0,dim_forecasters
1,dim_questions
2,fct_forecasts
3,raw_forecaster_demographics
4,raw_forecasts_export
5,raw_question_tags


### For each question, how many forecasters provided a prediction, and what is the median predicted probability?

In [27]:
fct_forecasts_df = con.execute("SELECT * FROM fct_forecasts").fetchdf()
## How many unique forecasters have made forecasts for each question?
fct_forecasts_df.groupby('question_id')['forecaster_id'].nunique().reset_index()


,question_id,forecaster_id
0,Q001,10
1,Q002,10
2,Q003,9
3,Q004,9
4,Q005,10
5,Q006,10
6,Q007,10
7,Q008,9


In [28]:
## median prediction for each question
fct_forecasts_df.groupby('question_id')['prediction'].median().reset_index()

,question_id,prediction
0,Q001,27.0
1,Q002,72.0
2,Q003,63.5
3,Q004,18.0
4,Q005,15.0
5,Q006,55.0
6,Q007,39.0
7,Q008,34.0


### How does the average predicted probability differ between superforecasters and other forecaster types?

In [29]:
# taking forecaster_type from dim_forecasters table to calculate average prediction for each forecaster type
fct_forecasts_with_type_df = con.execute("SELECT f.*, d.forecaster_type FROM fct_forecasts f JOIN dim_forecasters d ON f.forecaster_id = d.forecaster_id").fetchdf()
fct_forecasts_with_type_df.head(1)

,forecast_id,question_id,forecaster_id,prediction,forecast_timestamp,rationale,forecaster_type
0,p000,Q001,F001,25,2025-01-15 09:30:00,AI progress is impressive but Humanity's Last Exam is specifically designed to be extremely difficult. Incremental gains on benchmarks don't easily extrapolate to near-perfect performance.,superforecaster


In [30]:
fct_forecasts_with_type_df.groupby('forecaster_type')['prediction'].mean().reset_index()

,forecaster_type,prediction
0,expert,38.461538
1,public,40.647059
2,superforecaster,41.766667


does not seems to differ much